In [2]:
# =================================================================
# 최종 통합 스크립트
# 1. Mask R-CNN으로 장기(Organ) 위치 탐지 모델 학습
# 2. 학습된 모델로 이미지 시퀀스에서 장기 위치 추적
# 3. 위치 변화를 기반으로 카메라 이동(x, y, z)을 추정하여 JSON으로 저장
# =================================================================
import json
from pathlib import Path
import os
import cv2
import numpy as np
import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from PIL import Image, ImageDraw, ImageFile
from tqdm import tqdm

# PIL에서 이미지 파일이 잘렸을 때 발생하는 오류 방지
ImageFile.LOAD_TRUNCATED_IMAGES = True

# =================================================================
# PART 1: 장기 탐지 모델 학습 (Organ Detection Model Training)
# =================================================================

class OrganDataset(Dataset):
    """
    장기 이미지와 폴리곤 JSON 라벨을 Mask R-CNN 학습 형식으로 변환하는 데이터셋 클래스.
    SurgicalDatasetParser의 원칙과 제공된 JSON 구조를 결합하여 안정성을 높였습니다.
    """
    def __init__(self, image_dir: Path, json_dir: Path, transforms=None):
        self.image_dir = image_dir
        self.json_dir = json_dir
        self.transforms = transforms
        self.items = []

        print("데이터셋 유효성 검증 및 항목 준비 중...")
        json_files = sorted(list(self.json_dir.glob("*.json")))

        for json_file in tqdm(json_files, desc="JSON 파일 처리"):
            image_id = json_file.stem + ".png" # 또는 .jpg 등 이미지 확장자에 맞게 수정
            image_path = self.image_dir / image_id

            if not image_path.exists():
                continue

            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)

            # "annotations"가 비어있는 경우 건너뜀
            if not data.get("annotations"):
                continue

            # 이 이미지에 대한 모든 유효한 어노테이션(장기)을 수집
            valid_objects = []
            for ann in data["annotations"]:
                if "points" not in ann or len(ann["points"]) < 3:
                    continue

                try:
                    with Image.open(image_path) as img:
                        w, h = img.size
                except Exception as e:
                    print(f"경고: 이미지 파일을 열 수 없습니다. {image_path}, 오류: {e}")
                    continue

                coords = [(float(p[0]), float(p[1])) for p in ann["points"]]
                xs = [x for x, y in coords]
                ys = [y for x, y in coords]
                
                x_min, x_max = max(0, min(xs)), min(w, max(xs))
                y_min, y_max = max(0, min(ys)), min(h, max(ys))

                if not (x_max > x_min and y_max > y_min):
                    continue

                # 빈 마스크 검증
                mask_img = Image.new("L", (w, h), 0)
                ImageDraw.Draw(mask_img).polygon(coords, outline=1, fill=1)
                mask_array = np.array(mask_img)
                if not np.any(mask_array > 0):
                    continue
                
                valid_objects.append({
                    "points": coords,
                    "box": [x_min, y_min, x_max, y_max],
                    "mask": mask_array
                })

            if valid_objects:
                self.items.append({
                    "image_path": str(image_path),
                    "width": w,
                    "height": h,
                    "objects": valid_objects
                })

        print(f"총 {len(self.items)}개의 유효한 이미지-어노테이션 쌍을 준비했습니다.")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        image = Image.open(item['image_path']).convert("RGB")

        boxes, masks, labels = [], [], []
        for obj in item['objects']:
            boxes.append(obj['box'])
            masks.append(obj['mask'])
            labels.append(1) # 단일 클래스 'Liver'는 1로 통일

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "masks": torch.as_tensor(np.array(masks), dtype=torch.uint8)
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target

def get_organ_detector_model(num_classes=2): # 배경(0) + Liver(1)
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")
    in_features_box = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    return model

def train_organ_detector(image_dir, json_dir, model_save_path, num_epochs=10, batch_size=2):
    """장기 탐지 모델을 학습하고 지정된 경로에 저장합니다."""
    print("\n--- 파트 1: 장기 탐지 모델 학습 시작 ---")
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    
    dataset = OrganDataset(Path(image_dir), Path(json_dir), transforms=T.ToTensor())
    if len(dataset) == 0:
        print("오류: 학습할 유효 데이터가 없습니다. 경로와 JSON 내용을 확인하세요.")
        return False
        
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    def collate_fn(batch):
        return tuple(zip(*batch))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    
    model = get_organ_detector_model()
    model.to(device)
    
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            
            if not torch.isfinite(losses):
                continue
            
            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
            total_loss += losses.item()
        
        print(f"Epoch {epoch+1} 완료, 평균 Loss: {total_loss / len(train_loader):.4f}")

    Path(model_save_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), model_save_path)
    print(f"모델 저장 완료: {model_save_path}")
    return True

# =================================================================
# PART 2: 카메라 이동 방향 예측 (Camera Motion Prediction)
# =================================================================

class OrganTracker:
    """학습된 모델을 사용하여 이미지에서 장기를 탐지하는 추론기"""
    def __init__(self, model_path, device):
        self.device = device
        self.model = get_organ_detector_model()
        self.model.load_state_dict(torch.load(model_path, map_location=device))
        self.model.to(device)
        self.model.eval()
        self.transform = T.Compose([T.ToTensor()])

    @torch.no_grad()
    def detect_organ(self, image_path):
        image = Image.open(image_path).convert("RGB")
        image_tensor = self.transform(image).unsqueeze(0).to(self.device)
        prediction = self.model(image_tensor)
        
        # 가장 신뢰도 높은 객체 하나만 선택
        scores = prediction[0]['scores']
        if len(scores) == 0:
            return None # 탐지된 객체 없음
            
        best_idx = torch.argmax(scores)
        box = prediction[0]['boxes'][best_idx].cpu().numpy()
        return box # [x_min, y_min, x_max, y_max]

def predict_camera_motion(image_dir, model_path, output_json_path):
    """
    이미지 시퀀스를 처리하여 장기 위치를 추적하고,
    그 변화를 기반으로 카메라 이동을 예측하여 JSON 파일로 저장합니다.
    """
    print("\n--- 파트 2: 카메라 이동 방향 예측 시작 ---")
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    tracker = OrganTracker(model_path, device)
    
    image_files = sorted([p for p in Path(image_dir).glob("*.png")]) # 확장자 확인
    if len(image_files) < 2:
        print("오류: 이동을 예측하려면 2개 이상의 이미지가 필요합니다.")
        return

    previous_box = None
    motion_results = []
    
    for i, img_path in tqdm(enumerate(image_files), desc="이미지 처리 및 이동 예측", total=len(image_files)):
        current_box = tracker.detect_organ(img_path)
        
        dx, dy, dz = None, None, None
        
        if current_box is not None and previous_box is not None:
            # 중심점 계산
            prev_center_x = (previous_box[0] + previous_box[2]) / 2
            prev_center_y = (previous_box[1] + previous_box[3]) / 2
            curr_center_x = (current_box[0] + current_box[2]) / 2
            curr_center_y = (current_box[1] + current_box[3]) / 2
            
            # 면적 계산 (z축 이동의 근사치로 사용)
            prev_area = (previous_box[2] - previous_box[0]) * (previous_box[3] - previous_box[1])
            curr_area = (current_box[2] - current_box[0]) * (current_box[3] - current_box[1])

            # 이동 방향 계산
            dx = curr_center_x - prev_center_x
            dy = curr_center_y - prev_center_y
            dz = curr_area - prev_area
            
        motion_results.append({
            "frame_index": i,
            "image_id": img_path.name,
            "detected_box": current_box.tolist() if current_box is not None else None,
            "motion_from_prev": {
                "dx": dx, "dy": dy, "dz_proxy_area_change": dz
            }
        })
        previous_box = current_box

    # =================================================================
    # PART 3: 요구사항에 맞는 JSON 파일로 저장
    # =================================================================
    final_output = {
        "images": [p.name for p in image_files],
        "meta": {
            "description": "Camera motion prediction based on organ tracking.",
            "model_used": model_path,
            "dx_dy_unit": "pixels",
            "dz_proxy_unit": "pixel_area_change"
        },
        "annotations": motion_results
    }
    
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(final_output, f, indent=2, ensure_ascii=False)
    
    print(f"카메라 이동 예측 결과 저장 완료: {output_json_path}")


# =================================================================
# 메인 실행 블록
# =================================================================
if __name__ == '__main__':
    # --- 사용자 설정 변수 ---
    # 실제 환경에 맞게 이 경로들을 수정해주세요.
    IMAGE_DATA_DIR = "../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간"
    JSON_LABEL_DIR   = "../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/02.라벨링데이터/TL_2.장기 데이터_01.간"

    
    # 생성될 파일 경로
    MODEL_SAVE_PATH = "./organ_detector_final.pth"
    MOTION_OUTPUT_PATH = "./camera_motion_prediction.json"
    
    # --- 실행 파이프라인 ---
    # 1. 장기 탐지 모델 학습
    # 학습이 성공적으로 완료되었을 때만 다음 단계로 진행합니다.
    # 이미 학습된 모델이 있다면 이 부분을 주석 처리하고 바로 2단계로 넘어갈 수 있습니다.
    is_training_successful = train_organ_detector(
        image_dir=IMAGE_DATA_DIR,
        json_dir=JSON_LABEL_DIR,
        model_save_path=MODEL_SAVE_PATH,
        num_epochs=10 # 실제 학습 시에는 더 많은 epoch 필요
    )

    # 2. 학습된 모델로 카메라 이동 예측
    if is_training_successful and Path(MODEL_SAVE_PATH).exists():
        predict_camera_motion(
            image_dir=IMAGE_DATA_DIR,
            model_path=MODEL_SAVE_PATH,
            output_json_path=MOTION_OUTPUT_PATH
        )
    else:
        print("모델 학습이 실패했거나 모델 파일이 없어 카메라 이동 예측을 건너뜁니다.")




--- 파트 1: 장기 탐지 모델 학습 시작 ---
데이터셋 유효성 검증 및 항목 준비 중...


JSON 파일 처리:  57%|██████████████▏          | 539/952 [00:23<00:18, 22.18it/s]

경고: 이미지 파일을 열 수 없습니다. ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00116654.png, 오류: cannot identify image file '/tf/dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00116654.png'
경고: 이미지 파일을 열 수 없습니다. ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00116654.png, 오류: cannot identify image file '/tf/dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00116654.png'
경고: 이미지 파일을 열 수 없습니다. ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00119137.png, 오류: cannot identify image file '/tf/dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Traini

JSON 파일 처리:  57%|██████████████▎          | 546/952 [00:23<00:17, 23.60it/s]

경고: 이미지 파일을 열 수 없습니다. ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00169488.png, 오류: cannot identify image file '/tf/dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00169488.png'
경고: 이미지 파일을 열 수 없습니다. ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00286475.png, 오류: cannot identify image file '/tf/dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00286475.png'
경고: 이미지 파일을 열 수 없습니다. ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00286475.png, 오류: cannot identify image file '/tf/dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Traini

...

Epoch 9 완료, 평균 Loss: 0.1676


Epoch 10/10: 100%|████████████████████████████| 398/398 [02:44<00:00,  2.42it/s]


Epoch 10 완료, 평균 Loss: 0.1575
모델 저장 완료: ./organ_detector_final.pth

--- 파트 2: 카메라 이동 방향 예측 시작 ---


이미지 처리 및 이동 예측:  56%|████████▍      | 536/952 [01:00<00:47,  8.81it/s]


UnidentifiedImageError: cannot identify image file '/tf/dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00116654.png'

In [6]:
# =================================================================
# 최종 통합 스크립트 (로컬 가중치 로딩 및 오류 처리 강화)
# 1. Mask R-CNN으로 장기(Organ) 위치 탐지 모델 학습 (로컬 COCO 가중치 활용)
# 2. 학습 후 검증 데이터셋으로 mAP 성능 평가
# 3. 학습된 모델로 카메라 이동(x, y, z)을 추정하여 JSON으로 저장
# =================================================================
import json
from pathlib import Path
import os
import cv2
import numpy as np
import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops import box_iou
from PIL import Image, ImageDraw, ImageFile, UnidentifiedImageError
from tqdm import tqdm

# PIL에서 이미지 파일이 잘렸을 때 발생하는 오류 방지
ImageFile.LOAD_TRUNCATED_IMAGES = True

# =================================================================
# PART 1: 장기 탐지 모델 학습 (Organ Detection Model Training)
# =================================================================
class OrganDataset(Dataset):
    """
    장기 이미지와 폴리곤 JSON 라벨을 Mask R-CNN 학습 형식으로 변환하는 데이터셋 클래스.
    """
    def __init__(self, image_dir: Path, json_dir: Path, transforms=None):
        self.image_dir = image_dir
        self.json_dir = json_dir
        self.transforms = transforms
        self.items = []
        print("데이터셋 유효성 검증 및 항목 준비 중...")
        json_files = sorted(list(self.json_dir.glob("*.json")))

        for json_file in tqdm(json_files, desc="JSON 파일 처리"):
            image_id = json_file.stem + ".png"
            image_path = self.image_dir / image_id

            if not image_path.exists():
                continue

            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)

            if not data.get("annotations"):
                continue

            valid_objects = []
            # --- [오류 처리 수정] 손상된 이미지 파일을 열 때 발생하는 오류를 방지 ---
            try:
                with Image.open(image_path) as img:
                    w, h = img.size
            except UnidentifiedImageError:
                print(f"경고: 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: {image_path}")
                continue
            except Exception as e:
                print(f"경고: 이미지 파일을 열 수 없습니다. {image_path}, 오류: {e}")
                continue
            # --------------------------------------------------------------------
                
            for ann in data["annotations"]:
                if "points" not in ann or len(ann["points"]) < 3:
                    continue
                coords = [(float(p[0]), float(p[1])) for p in ann["points"]]
                xs = [x for x, y in coords]
                ys = [y for x, y in coords]
                
                x_min, x_max = max(0, min(xs)), min(w, max(xs))
                y_min, y_max = max(0, min(ys)), min(h, max(ys))

                if not (x_max > x_min and y_max > y_min):
                    continue
                mask_img = Image.new("L", (w, h), 0)
                ImageDraw.Draw(mask_img).polygon(coords, outline=1, fill=1)
                mask_array = np.array(mask_img)

                if not np.any(mask_array > 0):
                    continue
                
                valid_objects.append({
                    "points": coords,
                    "box": [x_min, y_min, x_max, y_max],
                    "mask": mask_array
                })

            if valid_objects:
                self.items.append({
                    "image_path": str(image_path),
                    "width": w,
                    "height": h,
                    "objects": valid_objects
                })

        print(f"총 {len(self.items)}개의 유효한 이미지-어노테이션 쌍을 준비했습니다.")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        # --- [오류 처리 수정] __getitem__ 에서도 파일 열기 오류를 처리 ---
        try:
            image = Image.open(item['image_path']).convert("RGB")
        except (UnidentifiedImageError, FileNotFoundError) as e:
            print(f"경고: __getitem__에서 이미지 로딩 실패. 건너뜁니다. {item['image_path']}, 오류: {e}")
            # 오류 발생 시 빈 텐서를 반환하여 collate_fn에서 걸러지도록 유도할 수 있으나,
            # 더 간단하게는, 이 오류가 발생하기 전에 __init__에서 걸러지므로 실제로는 거의 발생하지 않음.
            # 만약을 대비해 빈 데이터를 반환하도록 처리
            return None, None
        # -------------------------------------------------------------

        boxes, masks, labels = [], [], []
        for obj in item['objects']:
            boxes.append(obj['box'])
            masks.append(obj['mask'])
            labels.append(1) # 장기(organ) 클래스는 1로 통일

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "masks": torch.as_tensor(np.array(masks), dtype=torch.uint8)
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target

def get_organ_detector_model(num_classes=2, pretrained_coco_path: str = None):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None,weights_backbone=None)
    if pretrained_coco_path and Path(pretrained_coco_path).exists():
        print(f"로컬 COCO 사전 학습 가중치를 불러옵니다: {pretrained_coco_path}")
        state_dict = torch.load(pretrained_coco_path, map_location='cpu')
        filtered_sd = {k: v for k, v in state_dict.items() if not k.startswith(('roi_heads.box_predictor', 'roi_heads.mask_predictor'))}
        missing_keys, unexpected_keys = model.load_state_dict(filtered_sd, strict=False)
        print(f"가중치 로드 완료. 누락된 키: {len(missing_keys)}개, 예기치 않은 키: {len(unexpected_keys)}개")
    else:
        print("경고: 사전 학습된 COCO 가중치 없이 모델을 무작위로 초기화합니다.")

    in_features_box = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    return model

@torch.no_grad()
def evaluate_detector(model, data_loader, device, iou_threshold=0.5):
    model.eval()
    all_preds, all_targets = [], []
    for images, targets in tqdm(data_loader, desc="검증(Evaluation)"):
        images = [img.to(device) for img in images]
        predictions = model(images)
        all_preds.extend([{k: v.cpu() for k, v in p.items()} for p in predictions])
        all_targets.extend([{k: v.cpu() for k, v in t.items()} for t in targets])
        
    true_positives, false_positives, num_gt_boxes = 0, 0, 0
    for pred, target in zip(all_preds, all_targets):
        gt_boxes, pred_boxes = target['boxes'], pred['boxes']
        num_gt_boxes += len(gt_boxes)
        if len(pred_boxes) == 0: continue
        if len(gt_boxes) == 0:
            false_positives += len(pred_boxes)
            continue
        iou_matrix = box_iou(pred_boxes, gt_boxes)
        matched_gt = set()
        for i in range(len(pred_boxes)):
            max_iou, max_idx = iou_matrix[i].max(0)
            if max_iou >= iou_threshold and max_idx.item() not in matched_gt:
                true_positives += 1
                matched_gt.add(max_idx.item())
            else:
                false_positives += 1
                
    precision = true_positives / (true_positives + false_positives + 1e-6)
    recall = true_positives / (num_gt_boxes + 1e-6)
    print(f"\nIoU@ {iou_threshold:.2f} 기준, Precision: {precision:.4f}, Recall: {recall:.4f}")
    return precision, recall

def train_organ_detector(image_dir, json_dir, model_save_path, coco_weights_path, num_epochs=3, batch_size=2):
    print("\n--- 파트 1: 장기 탐지 모델 학습 시작 ---")
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    
    dataset = OrganDataset(Path(image_dir), Path(json_dir), transforms=T.Compose([T.ToTensor()]))
    if len(dataset) == 0:
        print("오류: 학습할 유효 데이터가 없습니다. 경로와 JSON 내용을 확인하세요.")
        return False
        
    indices = torch.randperm(len(dataset)).tolist()
    train_size = int(0.9 * len(dataset))
    train_dataset = torch.utils.data.Subset(dataset, indices[:train_size])
    val_dataset = torch.utils.data.Subset(dataset, indices[train_size:])

    def collate_fn(batch):
        batch = list(filter(lambda x: x is not None and x[0] is not None, batch))
        return tuple(zip(*batch)) if batch else ([], [])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

    model = get_organ_detector_model(pretrained_coco_path=coco_weights_path)
    model.to(device)
    
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [학습]"):
            if not images: continue
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            if not torch.isfinite(losses): continue
            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
            total_loss += losses.item()
        
        avg_train_loss = total_loss / len(train_loader) if len(train_loader) > 0 else 0
        print(f"\nEpoch {epoch+1} 완료, 평균 학습 Loss: {avg_train_loss:.4f}")
        
        if len(val_loader) > 0:
            evaluate_detector(model, val_loader, device=device)

    Path(model_save_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), model_save_path)
    print(f"\n모델 저장 완료: {model_save_path}")
    return True

# =================================================================
# PART 2 & 3: 카메라 이동 예측 및 JSON 저장
# =================================================================
class OrganTracker:
    def __init__(self, model_path, device):
        self.device = device
        self.model = get_organ_detector_model()
        self.model.load_state_dict(torch.load(model_path, map_location=device))
        self.model.to(device)
        self.model.eval()
        self.transform = T.Compose([T.ToTensor()])

    @torch.no_grad()
    def detect_organ(self, image_path):
        # --- [오류 처리 수정] 추론 시에도 손상된 이미지를 안전하게 건너뜀 ---
        try:
            image = Image.open(image_path).convert("RGB")
        except UnidentifiedImageError:
            print(f"\n[경고] 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: {image_path}")
            return None
        except Exception as e:
            print(f"\n[경고] 이미지 파일 처리 중 알 수 없는 오류. {image_path}, 오류: {e}")
            return None
        # -----------------------------------------------------------------
        image_tensor = self.transform(image).unsqueeze(0).to(self.device)
        prediction = self.model(image_tensor)
        scores = prediction[0]['scores']
        if len(scores) == 0: return None
        best_idx = torch.argmax(scores)
        box = prediction[0]['boxes'][best_idx].cpu().numpy()
        return box

def predict_camera_motion(image_dir, model_path, output_json_path):
    print("\n--- 파트 2 & 3: 카메라 이동 방향 예측 및 JSON 저장 시작 ---")
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    tracker = OrganTracker(model_path, device)
    
    image_files = sorted([p for p in Path(image_dir).glob("*.png")])
    if len(image_files) < 2:
        print("오류: 이동을 예측하려면 2개 이상의 이미지가 필요합니다.")
        return

    previous_box = None
    motion_results = []
    
    for i, img_path in tqdm(enumerate(image_files), desc="이미지 처리 및 이동 예측", total=len(image_files)):
        current_box = tracker.detect_organ(img_path)
        dx, dy, dz = None, None, None
        
        if current_box is not None and previous_box is not None:
            prev_center_x, prev_center_y = (previous_box[0] + previous_box[2]) / 2, (previous_box[1] + previous_box[3]) / 2
            curr_center_x, curr_center_y = (current_box[0] + current_box[2]) / 2, (current_box[1] + current_box[3]) / 2
            prev_area = (previous_box[2] - previous_box[0]) * (previous_box[3] - previous_box[1])
            curr_area = (current_box[2] - current_box[0]) * (current_box[3] - current_box[1])
            dx, dy, dz = curr_center_x - prev_center_x, curr_center_y - prev_center_y, curr_area - prev_area
            
        motion_results.append({
            "frame_index": i, "image_id": img_path.name,
            "detected_box": current_box.tolist() if current_box is not None else None,
            "motion_from_prev": {
                "dx": float(dx) if dx is not None else None,
                "dy": float(dy) if dy is not None else None,
                "dz_proxy_area_change": float(dz) if dz is not None else None
            }
        })
        previous_box = current_box

    final_output = {
        "images": [p.name for p in image_files],
        "meta": {"description": "Camera motion prediction based on organ tracking."},
        "annotations": motion_results
    }
    
    Path(output_json_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(final_output, f, indent=2, ensure_ascii=False)
    
    print(f"카메라 이동 예측 결과 저장 완료: {output_json_path}")

# =================================================================
# 메인 실행 블록
# =================================================================
if __name__ == '__main__':
    IMAGE_DATA_DIR = "../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간"
    JSON_LABEL_DIR   = "../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/02.라벨링데이터/TL_2.장기 데이터_01.간"
    MODEL_SAVE_PATH = "./organ_detector_final_coco.pth"
    MOTION_OUTPUT_PATH = "./camera_motion_prediction_coco.json"
    
    COCO_WEIGHTS_PATH = "/root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth"
    
    is_training_successful = train_organ_detector(
        image_dir=IMAGE_DATA_DIR,
        json_dir=JSON_LABEL_DIR,
        model_save_path=MODEL_SAVE_PATH,
        coco_weights_path=COCO_WEIGHTS_PATH,
        num_epochs=3
    )

    if is_training_successful and Path(MODEL_SAVE_PATH).exists():
        predict_camera_motion(
            image_dir=IMAGE_DATA_DIR,
            model_path=MODEL_SAVE_PATH,
            output_json_path=MOTION_OUTPUT_PATH
        )
    else:
        print("모델 학습이 실패했거나 모델 파일이 없어 카메라 이동 예측을 건너뜁니다.")



--- 파트 1: 장기 탐지 모델 학습 시작 ---
데이터셋 유효성 검증 및 항목 준비 중...


JSON 파일 처리:  56%|██████████████           | 537/952 [00:17<00:15, 26.89it/s]

경고: 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00116654.png
경고: 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00119137.png
경고: 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00119550.png
경고: 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00119962.png


JSON 파일 처리:  57%|██████████████▎          | 545/952 [00:17<00:15, 25.83it/s]

경고: 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00125131.png
경고: 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00169488.png
경고: 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00286475.png
경고: 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00325677.png
경고: 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0055_00123070.png
경고: 손상된 이미

...


IoU@ 0.50 기준, Precision: 0.0576, Recall: 0.9433

모델 저장 완료: ./organ_detector_final_coco.pth

--- 파트 2 & 3: 카메라 이동 방향 예측 및 JSON 저장 시작 ---
경고: 사전 학습된 COCO 가중치 없이 모델을 무작위로 초기화합니다.


이미지 처리 및 이동 예측:  64%|████████▉     | 605/952 [00:59<00:02, 138.32it/s]


[경고] 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00116654.png

[경고] 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00119137.png

[경고] 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00119550.png

[경고] 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00119962.png

[경고] 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: ../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_01.간/SNUH_DC16_PJW0_RHC0_0047_00125131.png


이미지 처리 및 이동 예측: 100%|███████████████| 952/952 [01:52<00:00,  8.45it/s]

카메라 이동 예측 결과 저장 완료: ./camera_motion_prediction_coco.json


In [1]:
# =================================================================
# 최종 통합 스크립트 (로컬 가중치 로딩 및 오류 처리 강화)
# 1. Mask R-CNN으로 장기(Organ) 위치 탐지 모델 학습 (로컬 COCO 가중치 활용)
# 2. 학습 후 검증 데이터셋으로 mAP 성능 평가
# 3. 학습된 모델로 카메라 이동(x, y, z)을 추정하여 JSON으로 저장
# =================================================================
import json
from pathlib import Path
import os
import cv2
import numpy as np
import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops import box_iou
from PIL import Image, ImageDraw, ImageFile, UnidentifiedImageError
from tqdm import tqdm

# PIL에서 이미지 파일이 잘렸을 때 발생하는 오류 방지
ImageFile.LOAD_TRUNCATED_IMAGES = True

# =================================================================
# PART 1: 장기 탐지 모델 학습 (Organ Detection Model Training)
# =================================================================
class OrganDataset(Dataset):
    """
    장기 이미지와 폴리곤 JSON 라벨을 Mask R-CNN 학습 형식으로 변환하는 데이터셋 클래스.
    """
    def __init__(self, image_dir: Path, json_dir: Path, transforms=None):
        self.image_dir = image_dir
        self.json_dir = json_dir
        self.transforms = transforms
        self.items = []
        print("데이터셋 유효성 검증 및 항목 준비 중...")
        json_files = sorted(list(self.json_dir.glob("*.json")))

        for json_file in tqdm(json_files, desc="JSON 파일 처리"):
            image_id = json_file.stem + ".png"
            image_path = self.image_dir / image_id

            if not image_path.exists():
                continue

            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)

            if not data.get("annotations"):
                continue

            valid_objects = []
            # --- [오류 처리 수정] 손상된 이미지 파일을 열 때 발생하는 오류를 방지 ---
            try:
                with Image.open(image_path) as img:
                    w, h = img.size
            except UnidentifiedImageError:
                print(f"경고: 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: {image_path}")
                continue
            except Exception as e:
                print(f"경고: 이미지 파일을 열 수 없습니다. {image_path}, 오류: {e}")
                continue
            # --------------------------------------------------------------------
                
            for ann in data["annotations"]:
                if "points" not in ann or len(ann["points"]) < 3:
                    continue
                coords = [(float(p[0]), float(p[1])) for p in ann["points"]]
                xs = [x for x, y in coords]
                ys = [y for x, y in coords]
                
                x_min, x_max = max(0, min(xs)), min(w, max(xs))
                y_min, y_max = max(0, min(ys)), min(h, max(ys))

                if not (x_max > x_min and y_max > y_min):
                    continue
                mask_img = Image.new("L", (w, h), 0)
                ImageDraw.Draw(mask_img).polygon(coords, outline=1, fill=1)
                mask_array = np.array(mask_img)

                if not np.any(mask_array > 0):
                    continue
                
                valid_objects.append({
                    "points": coords,
                    "box": [x_min, y_min, x_max, y_max],
                    "mask": mask_array
                })

            if valid_objects:
                self.items.append({
                    "image_path": str(image_path),
                    "width": w,
                    "height": h,
                    "objects": valid_objects
                })

        print(f"총 {len(self.items)}개의 유효한 이미지-어노테이션 쌍을 준비했습니다.")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        # --- [오류 처리 수정] __getitem__ 에서도 파일 열기 오류를 처리 ---
        try:
            image = Image.open(item['image_path']).convert("RGB")
        except (UnidentifiedImageError, FileNotFoundError) as e:
            print(f"경고: __getitem__에서 이미지 로딩 실패. 건너뜁니다. {item['image_path']}, 오류: {e}")
            # 오류 발생 시 빈 텐서를 반환하여 collate_fn에서 걸러지도록 유도할 수 있으나,
            # 더 간단하게는, 이 오류가 발생하기 전에 __init__에서 걸러지므로 실제로는 거의 발생하지 않음.
            # 만약을 대비해 빈 데이터를 반환하도록 처리
            return None, None
        # -------------------------------------------------------------

        boxes, masks, labels = [], [], []
        for obj in item['objects']:
            boxes.append(obj['box'])
            masks.append(obj['mask'])
            labels.append(1) # 장기(organ) 클래스는 1로 통일

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "masks": torch.as_tensor(np.array(masks), dtype=torch.uint8)
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target

def get_organ_detector_model(num_classes=2, pretrained_coco_path: str = None):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None,weights_backbone=None)
    if pretrained_coco_path and Path(pretrained_coco_path).exists():
        print(f"로컬 COCO 사전 학습 가중치를 불러옵니다: {pretrained_coco_path}")
        state_dict = torch.load(pretrained_coco_path, map_location='cpu')
        filtered_sd = {k: v for k, v in state_dict.items() if not k.startswith(('roi_heads.box_predictor', 'roi_heads.mask_predictor'))}
        missing_keys, unexpected_keys = model.load_state_dict(filtered_sd, strict=False)
        print(f"가중치 로드 완료. 누락된 키: {len(missing_keys)}개, 예기치 않은 키: {len(unexpected_keys)}개")
    else:
        print("경고: 사전 학습된 COCO 가중치 없이 모델을 무작위로 초기화합니다.")

    in_features_box = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    return model

@torch.no_grad()
def evaluate_detector(model, data_loader, device, iou_threshold=0.5):
    model.eval()
    all_preds, all_targets = [], []
    for images, targets in tqdm(data_loader, desc="검증(Evaluation)"):
        images = [img.to(device) for img in images]
        predictions = model(images)
        all_preds.extend([{k: v.cpu() for k, v in p.items()} for p in predictions])
        all_targets.extend([{k: v.cpu() for k, v in t.items()} for t in targets])
        
    true_positives, false_positives, num_gt_boxes = 0, 0, 0
    for pred, target in zip(all_preds, all_targets):
        gt_boxes, pred_boxes = target['boxes'], pred['boxes']
        num_gt_boxes += len(gt_boxes)
        if len(pred_boxes) == 0: continue
        if len(gt_boxes) == 0:
            false_positives += len(pred_boxes)
            continue
        iou_matrix = box_iou(pred_boxes, gt_boxes)
        matched_gt = set()
        for i in range(len(pred_boxes)):
            max_iou, max_idx = iou_matrix[i].max(0)
            if max_iou >= iou_threshold and max_idx.item() not in matched_gt:
                true_positives += 1
                matched_gt.add(max_idx.item())
            else:
                false_positives += 1
                
    precision = true_positives / (true_positives + false_positives + 1e-6)
    recall = true_positives / (num_gt_boxes + 1e-6)
    print(f"\nIoU@ {iou_threshold:.2f} 기준, Precision: {precision:.4f}, Recall: {recall:.4f}")
    return precision, recall

def train_organ_detector(image_dir, json_dir, model_save_path, coco_weights_path, log_save_path, num_epochs=10, batch_size=2):
    print("\n--- 파트 1: 장기 탐지 모델 학습 시작 ---")
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    
    dataset = OrganDataset(Path(image_dir), Path(json_dir), transforms=T.Compose([T.ToTensor()]))
    if len(dataset) == 0:
        print("오류: 학습할 유효 데이터가 없습니다. 경로와 JSON 내용을 확인하세요.")
        return False
        
    indices = torch.randperm(len(dataset)).tolist()
    train_size = int(0.9 * len(dataset))
    train_dataset = torch.utils.data.Subset(dataset, indices[:train_size])
    val_dataset = torch.utils.data.Subset(dataset, indices[train_size:])

    def collate_fn(batch):
        batch = list(filter(lambda x: x is not None and x[0] is not None, batch))
        return tuple(zip(*batch)) if batch else ([], [])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

    model = get_organ_detector_model(pretrained_coco_path=coco_weights_path)
    model.to(device)
    
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

    # [수정] 학습 기록을 저장할 리스트
    training_history = []
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [학습]"):
            if not images: continue
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            if not torch.isfinite(losses): continue
            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
            total_loss += losses.item()
        
        avg_train_loss = total_loss / len(train_loader) if len(train_loader) > 0 else 0
        print(f"\nEpoch {epoch+1} 완료, 평균 학습 Loss: {avg_train_loss:.4f}")
            
        precision, recall = 0.0, 0.0
        if len(val_loader) > 0:
            precision, recall = evaluate_detector(model, val_loader, device=device)
    
        # [수정] 현재 에폭의 결과를 딕셔너리로 저장
        epoch_log = {
            'epoch': epoch + 1,
            'avg_train_loss': avg_train_loss,
            'precision': precision,
            'recall': recall
        }
        training_history.append(epoch_log)

    Path(model_save_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), model_save_path)
    print(f"\n모델 저장 완료: {model_save_path}")
    
    if log_save_path:
        Path(log_save_path).parent.mkdir(parents=True, exist_ok=True)
        with open(log_save_path, 'w', encoding='utf-8') as f:
            json.dump(training_history, f, indent=2, ensure_ascii=False)
        print(f"학습 로그 저장 완료: {log_save_path}")
        
    return True

# =================================================================
# PART 2 & 3: 카메라 이동 예측 및 JSON 저장
# =================================================================
class OrganTracker:
    def __init__(self, model_path, device):
        self.device = device
        self.model = get_organ_detector_model()
        self.model.load_state_dict(torch.load(model_path, map_location=device))
        self.model.to(device)
        self.model.eval()
        self.transform = T.Compose([T.ToTensor()])

    @torch.no_grad()
    def detect_organ(self, image_path):
        # --- [오류 처리 수정] 추론 시에도 손상된 이미지를 안전하게 건너뜀 ---
        try:
            image = Image.open(image_path).convert("RGB")
        except UnidentifiedImageError:
            print(f"\n[경고] 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: {image_path}")
            return None
        except Exception as e:
            print(f"\n[경고] 이미지 파일 처리 중 알 수 없는 오류. {image_path}, 오류: {e}")
            return None
        # -----------------------------------------------------------------
        image_tensor = self.transform(image).unsqueeze(0).to(self.device)
        prediction = self.model(image_tensor)
        scores = prediction[0]['scores']
        if len(scores) == 0: return None
        best_idx = torch.argmax(scores)
        box = prediction[0]['boxes'][best_idx].cpu().numpy()
        return box

def predict_camera_motion(image_dir, model_path, output_json_path):
    print("\n--- 파트 2 & 3: 카메라 이동 방향 예측 및 JSON 저장 시작 ---")
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    tracker = OrganTracker(model_path, device)
    
    image_files = sorted([p for p in Path(image_dir).glob("*.png")])
    if len(image_files) < 2:
        print("오류: 이동을 예측하려면 2개 이상의 이미지가 필요합니다.")
        return

    previous_box = None
    motion_results = []
    
    for i, img_path in tqdm(enumerate(image_files), desc="이미지 처리 및 이동 예측", total=len(image_files)):
        current_box = tracker.detect_organ(img_path)
        dx, dy, dz = None, None, None
        
        if current_box is not None and previous_box is not None:
            prev_center_x, prev_center_y = (previous_box[0] + previous_box[2]) / 2, (previous_box[1] + previous_box[3]) / 2
            curr_center_x, curr_center_y = (current_box[0] + current_box[2]) / 2, (current_box[1] + current_box[3]) / 2
            prev_area = (previous_box[2] - previous_box[0]) * (previous_box[3] - previous_box[1])
            curr_area = (current_box[2] - current_box[0]) * (current_box[3] - current_box[1])
            dx, dy, dz = curr_center_x - prev_center_x, curr_center_y - prev_center_y, curr_area - prev_area
            
        motion_results.append({
            "frame_index": i, "image_id": img_path.name,
            "detected_box": current_box.tolist() if current_box is not None else None,
            "motion_from_prev": {
                "dx": float(dx) if dx is not None else None,
                "dy": float(dy) if dy is not None else None,
                "dz_proxy_area_change": float(dz) if dz is not None else None
            }
        })
        previous_box = current_box

    final_output = {
        "images": [p.name for p in image_files],
        "meta": {"description": "Camera motion prediction based on organ tracking."},
        "annotations": motion_results
    }
    
    Path(output_json_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(final_output, f, indent=2, ensure_ascii=False)
    
    print(f"카메라 이동 예측 결과 저장 완료: {output_json_path}")

# =================================================================
# 메인 실행 블록
# =================================================================
if __name__ == '__main__':
    IMAGE_DATA_DIR = "../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/01.원천데이터/TS_2.장기 데이터_03.비장"
    JSON_LABEL_DIR   = "../../dataset/dataset002/281.6개 중대암수술의 첨단 AI기술개발을 위한 단계학습데이터셋 개발/01-1.정식개방데이터/Training/02.라벨링데이터/TL_2.장기 데이터_03.비장"
    MODEL_SAVE_PATH = "./organ_detector_final_coco3.pth"
    MOTION_OUTPUT_PATH = "./camera_motion_prediction_coco3.json"
    
    TRAINING_LOG_PATH = "./training_log3.json"
    
    COCO_WEIGHTS_PATH = "/root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth"
    
    is_training_successful = train_organ_detector(
        image_dir=IMAGE_DATA_DIR,
        json_dir=JSON_LABEL_DIR,
        model_save_path=MODEL_SAVE_PATH,
        coco_weights_path=COCO_WEIGHTS_PATH,
        log_save_path=TRAINING_LOG_PATH,
        num_epochs=10
    )

    if is_training_successful and Path(MODEL_SAVE_PATH).exists():
        predict_camera_motion(
            image_dir=IMAGE_DATA_DIR,
            model_path=MODEL_SAVE_PATH,
            output_json_path=MOTION_OUTPUT_PATH
        )
    else:
        print("모델 학습이 실패했거나 모델 파일이 없어 카메라 이동 예측을 건너뜁니다.")



--- 파트 1: 장기 탐지 모델 학습 시작 ---
데이터셋 유효성 검증 및 항목 준비 중...


JSON 파일 처리: 100%|█████████████████████████| 756/756 [04:21<00:00,  2.89it/s]


총 756개의 유효한 이미지-어노테이션 쌍을 준비했습니다.
로컬 COCO 사전 학습 가중치를 불러옵니다: /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth
가중치 로드 완료. 누락된 키: 24개, 예기치 않은 키: 88개


Epoch 1/10 [학습]: 100%|██████████████████████| 340/340 [03:30<00:00,  1.61it/s]



Epoch 1 완료, 평균 학습 Loss: 0.6026


...


Epoch 10 완료, 평균 학습 Loss: 0.1430


검증(Evaluation): 100%|█████████████████████████| 76/76 [00:11<00:00,  6.72it/s]



IoU@ 0.50 기준, Precision: 0.4478, Recall: 0.9890

모델 저장 완료: ./organ_detector_final_coco3.pth
학습 로그 저장 완료: ./training_log3.json

--- 파트 2 & 3: 카메라 이동 방향 예측 및 JSON 저장 시작 ---
경고: 사전 학습된 COCO 가중치 없이 모델을 무작위로 초기화합니다.


이미지 처리 및 이동 예측: 100%|███████████████| 756/756 [01:18<00:00,  9.59it/s]

카메라 이동 예측 결과 저장 완료: ./camera_motion_prediction_coco3.json


In [ ]:
--- 파트 1: 장기 탐지 모델 학습 시작 ---
데이터셋 유효성 검증 및 항목 준비 중...
JSON 파일 처리: 100%|█████████████████████████| 756/756 [04:21<00:00,  2.89it/s]
총 756개의 유효한 이미지-어노테이션 쌍을 준비했습니다.
로컬 COCO 사전 학습 가중치를 불러옵니다: /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth
가중치 로드 완료. 누락된 키: 24개, 예기치 않은 키: 88개
Epoch 1/10 [학습]: 100%|██████████████████████| 340/340 [03:30<00:00,  1.61it/s]

Epoch 1 완료, 평균 학습 Loss: 0.6026
검증(Evaluation): 100%|█████████████████████████| 76/76 [00:35<00:00,  2.17it/s]

IoU@ 0.50 기준, Precision: 0.0285, Recall: 0.9451
Epoch 2/10 [학습]: 100%|██████████████████████| 340/340 [02:33<00:00,  2.21it/s]

Epoch 2 완료, 평균 학습 Loss: 0.3404
검증(Evaluation): 100%|█████████████████████████| 76/76 [00:20<00:00,  3.72it/s]

IoU@ 0.50 기준, Precision: 0.0912, Recall: 0.9780
Epoch 3/10 [학습]: 100%|██████████████████████| 340/340 [02:33<00:00,  2.21it/s]

Epoch 3 완료, 평균 학습 Loss: 0.2663
검증(Evaluation): 100%|█████████████████████████| 76/76 [00:13<00:00,  5.53it/s]

IoU@ 0.50 기준, Precision: 0.1517, Recall: 0.9670
Epoch 4/10 [학습]: 100%|██████████████████████| 340/340 [02:32<00:00,  2.23it/s]

Epoch 4 완료, 평균 학습 Loss: 0.2328
검증(Evaluation): 100%|█████████████████████████| 76/76 [00:15<00:00,  4.88it/s]

IoU@ 0.50 기준, Precision: 0.1426, Recall: 0.9670
Epoch 5/10 [학습]: 100%|██████████████████████| 340/340 [02:35<00:00,  2.19it/s]

Epoch 5 완료, 평균 학습 Loss: 0.2313
검증(Evaluation): 100%|█████████████████████████| 76/76 [00:14<00:00,  5.23it/s]

IoU@ 0.50 기준, Precision: 0.1528, Recall: 0.9890
Epoch 6/10 [학습]: 100%|██████████████████████| 340/340 [02:31<00:00,  2.24it/s]

Epoch 6 완료, 평균 학습 Loss: 0.1899
검증(Evaluation): 100%|█████████████████████████| 76/76 [00:11<00:00,  6.66it/s]

IoU@ 0.50 기준, Precision: 0.3560, Recall: 0.9780
Epoch 7/10 [학습]: 100%|██████████████████████| 340/340 [02:33<00:00,  2.22it/s]

Epoch 7 완료, 평균 학습 Loss: 0.1691
검증(Evaluation): 100%|█████████████████████████| 76/76 [00:12<00:00,  6.28it/s]

IoU@ 0.50 기준, Precision: 0.3689, Recall: 0.9890
Epoch 8/10 [학습]: 100%|██████████████████████| 340/340 [02:35<00:00,  2.19it/s]

Epoch 8 완료, 평균 학습 Loss: 0.1593
검증(Evaluation): 100%|█████████████████████████| 76/76 [00:11<00:00,  6.90it/s]

IoU@ 0.50 기준, Precision: 0.3846, Recall: 0.9890
Epoch 9/10 [학습]: 100%|██████████████████████| 340/340 [02:30<00:00,  2.27it/s]

Epoch 9 완료, 평균 학습 Loss: 0.1488
검증(Evaluation): 100%|█████████████████████████| 76/76 [00:11<00:00,  6.88it/s]

IoU@ 0.50 기준, Precision: 0.3930, Recall: 0.9890
Epoch 10/10 [학습]: 100%|█████████████████████| 340/340 [02:31<00:00,  2.24it/s]

Epoch 10 완료, 평균 학습 Loss: 0.1430
검증(Evaluation): 100%|█████████████████████████| 76/76 [00:11<00:00,  6.72it/s]

IoU@ 0.50 기준, Precision: 0.4478, Recall: 0.9890

모델 저장 완료: ./organ_detector_final_coco3.pth
학습 로그 저장 완료: ./training_log3.json

--- 파트 2 & 3: 카메라 이동 방향 예측 및 JSON 저장 시작 ---
경고: 사전 학습된 COCO 가중치 없이 모델을 무작위로 초기화합니다.
이미지 처리 및 이동 예측: 100%|███████████████| 756/756 [01:18<00:00,  9.59it/s]
카메라 이동 예측 결과 저장 완료: ./camera_motion_prediction_coco3.json